# Run the full ETL pipeline (Option 1)
`public` (source) -> `staging` -> `transformed` -> `clean`

**Before you start:** put `rules.sql`, `extract.sql`, `transform.sql`, `load.sql` and `schema.sql` in `SQL_DIR`, and the five CSV files in `DATA_DIR`.
Run the cells top to bottom. Step 2 is destructive for `public` (it reloads it from the CSVs), so it is guarded by `CONFIRM_RESET`.

In [ ]:
from pathlib import Path
from io import StringIO
import pandas as pd
from sqlalchemy import text
from connectdb import get_engine

engine = get_engine()

SQL_DIR  = Path("/home/joseph/Projects/Retail-Analytics-Data/etl")    # folder with rules.sql, extract.sql, transform.sql, load.sql, schema.sql
DATA_DIR = Path("/home/joseph/Projects/Retail-Analytics-Data/data")   # folder with fact_sales.csv, dim_*.csv

CONFIRM_RESET   = True   # set True to reload public from the CSVs (step 2)
LOAD_RAW_SALES  = True    # optional: load original sale categories, only used for the audit table

def run_file(path):
    raw = engine.raw_connection()
    try:
        raw.cursor().execute(Path(path).read_text())
        raw.commit()
    finally:
        raw.close()

def query(sql):
    return pd.read_sql(text(sql), engine)
print("connected:", query("select current_database() as db").iloc[0, 0])

connected: postgres


## Step 2: reset `public` from the CSVs
This removes the earlier notebook edits (category update, customer deletes, `customer_dedup_map`, `fact_sales_clean`) and the old pipeline schemas.

In [ ]:
if not CONFIRM_RESET:
    raise SystemExit("Set CONFIRM_RESET = True to reload public from the CSVs.")

raw = engine.raw_connection()
cur = raw.cursor()
cur.execute("""
    DROP SCHEMA IF EXISTS clean, transformed, staging, etl CASCADE;
    DROP VIEW  IF EXISTS public.fact_sales_clean;
    DROP TABLE IF EXISTS public.customer_dedup_map, public.fact_sales, public.dim_store,
                         public.dim_customer, public.dim_product, public.dim_date CASCADE;
""")
raw.commit()
cur.execute("SET search_path TO public;\n" + (DATA_DIR / "schema.sql").read_text())
raw.commit()
for table in ["dim_date", "dim_product", "dim_customer", "dim_store", "fact_sales"]:
    with open(DATA_DIR / f"{table}.csv") as fh:
        cur.copy_expert(f"COPY public.{table} FROM STDIN WITH CSV HEADER", fh)
raw.commit()
raw.close()
query("""select 'fact_sales' t, count(*) n from public.fact_sales union all
         select 'dim_product', count(*) from public.dim_product union all
         select 'dim_customer', count(*) from public.dim_customer""")

,t,n
0,fact_sales,60000
1,dim_product,400
2,dim_customer,2000


## Step 3: install the rules and the three stage functions

In [ ]:
for name in ["rules", "extract", "transform", "load"]:
    run_file(SQL_DIR / f"{name}.sql")
    print("installed", name)
query("select count(*) as product_names_mapped from etl.product_category_map")

installed rules
installed extract
installed transform
installed load


,product_names_mapped
0,42


## Step 4: run extract -> transform -> load

In [ ]:
raw = engine.raw_connection()
conn = getattr(raw, "driver_connection", None) or raw.connection
cur = raw.cursor()

cur.execute("SELECT etl.extract()"); raw.commit()

if LOAD_RAW_SALES:
    df = pd.read_csv(DATA_DIR / "fact_sales.csv", usecols=["sale_id", "category"])
    buf = StringIO(); df.to_csv(buf, index=False); buf.seek(0)
    cur.copy_expert("COPY staging.raw_sales (sale_id, category) FROM STDIN WITH CSV HEADER", buf)
    raw.commit()

del conn.notices[:]
cur.execute("SELECT etl.transform()"); raw.commit()
cur.execute("SELECT etl.load()");      raw.commit()

for n in conn.notices:
    if "Issue" in n:
        print(n.strip())
raw.close()
print("pipeline finished")

NOTICE:  Issue 3: source month_name mismatches = 667
NOTICE:  Issue 5: source products with unit_cost > list_price = 313
NOTICE:  Issue 8: sale category equals product category on 27.5 percent of rows
NOTICE:  Issue 7: stored revenue differs from the standard formula on 99.91 percent of rows
pipeline finished


## Step 5: validate

In [ ]:
query("""
select 'dim_customer rows (expect 1995)'          as check_name, count(*)::text as value from clean.dim_customer
union all select 'fact_sales rows (expect 60000)', count(*)::text from clean.fact_sales
union all select 'distinct sale categories (expect 4)', count(distinct category)::text from clean.fact_sales
union all select 'product_group values (expect 2)', count(distinct product_group)::text from clean.dim_product
union all select 'products without product_group (expect 0)', count(*)::text from clean.dim_product where product_group is null
union all select 'negative-margin products (expect 0)', count(*)::text from clean.dim_product where unit_cost > list_price
union all select 'month_name mismatches (expect 0)', count(*)::text
       from clean.dim_date where month_name <> to_char(full_date, 'FMMonth')
union all select 'sales without a quantity basis (expect 0)', count(*)::text from clean.fact_sales where quantity_basis is null
union all select 'sale category = product category, percent (info)',
       round(100.0 * count(*) filter (where category_matches_product) / count(*), 1)::text from clean.fact_sales
union all select 'revenue mismatch percent (documented)',
       round(100.0 * count(*) filter (where revenue_mismatch) / count(*), 2)::text from clean.fact_sales
""")

,check_name,value
0,dim_customer rows (expect 1995),1995
1,fact_sales rows (expect 60000),60000
2,distinct sale categories (expect 4),4
3,product_group values (expect 2),2
4,products without product_group (expect 0),0
5,negative-margin products (expect 0),0
6,month_name mismatches (expect 0),0
7,sales without a quantity basis (expect 0),0
8,"sale category = product category, percent (info)",27.5
9,revenue mismatch percent (documented),99.91


In [ ]:
query("""
select category, count(*) as sales, round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
from clean.fact_sales group by category order by sales desc
""")

,category,sales,pct
0,Electronics,25200,42.0
1,Home,15600,26.0
2,Apparel,12000,20.0
3,Grocery,7200,12.0


In [ ]:
query("""
select product_group, category as original_product_category, count(*) as products
from clean.dim_product group by 1, 2 order by 1, 3 desc
""")

,product_group,original_product_category,products
0,Electronics,Electronics,147
1,Electronics,Grocery,99
2,Electronics,Apparel,45
3,Home,Home,109


In [ ]:
query("""select quantity_basis, count(*) as sales_rows, sum(quantity) as sum_quantity,
       sum(units_sold) as sum_units_sold
from clean.fact_sales group by 1""")

,quantity_basis,sales_rows,sum_quantity,sum_units_sold
0,unit,56471,125662,125662
1,pack,3529,7981,31924


In [ ]:
query("""select count(*) as missing_basis from clean.fact_sales
where quantity_basis is null or units_sold is null;""")

,missing_basis
0,0


In [ ]:
query("""select (select count(*) from public.dim_date
        where month_name <> trim(to_char(full_date,'Month'))) as source_bad,
       (select count(*) from clean.dim_date
        where month_name <> to_char(full_date,'FMMonth')) as clean_bad;""")

,source_bad,clean_bad
0,667,0


In [ ]:
query("""select conname from pg_constraint where conname = 'chk_month_name';""")

,conname
0,chk_month_name


In [ ]:
query("""select discount_pct, count(*) from clean.fact_sales group by 1;""")

,discount_pct,count
0,0.35,60000


In [ ]:
query("""select issue_no, decision from etl.decision_log where issue_no = 4;""")

,issue_no,decision
0,4,accept


In [ ]:
query("""select conname from pg_constraint where conname = 'chk_discount';""")

,conname
0,chk_discount


In [ ]:
query("""select (select count(*) from public.dim_product where unit_cost > list_price) as source_negative,
       (select count(*) from clean.dim_product  where unit_cost > list_price) as clean_negative;""")

,source_negative,clean_negative
0,313,0


In [ ]:
query("""select min(margin_pct), max(margin_pct) from clean.dim_product;""")

,min,max
0,37.5,37.5


In [ ]:
query("""select issue_no, decision from etl.decision_log where issue_no = 5;""")

,issue_no,decision
0,5,fix


In [ ]:
query("""select (select count(*) from public.dim_customer) as customers_before,
       (select count(*) from clean.dim_customer)  as customers_after,
       (select count(*) from clean.customer_dedup_map) as merged_away;""")

,customers_before,customers_after,merged_away
0,2000,1995,5


In [ ]:
query("""select count(*) as pairs,
       count(*) filter (where merged) as merged_pairs,
       count(*) filter (where not same_name) as different_name_pairs
from clean.customer_duplicate_pairs;""")

,pairs,merged_pairs,different_name_pairs
0,15,5,10


In [ ]:
query("""select
 (select count(*) from clean.fact_sales f join clean.customer_dedup_map m
    on m.loser_key = f.customer_key) as sales_on_removed_customers,
 (select count(*) from clean.fact_sales f left join clean.dim_customer c
    using (customer_key) where c.customer_key is null) as orphan_sales,
 (select count(*) from (select 1 from clean.dim_customer
    group by email, lower(customer_name) having count(*) > 1) x) as remaining_duplicates,
 (select count(*) from clean.customer_review) as review_rows;""")


,sales_on_removed_customers,orphan_sales,remaining_duplicates,review_rows
0,0,0,0,20


In [ ]:
query("""select count(*) as total,
       count(*) filter (where revenue_mismatch) as mismatches,
       round(100.0*count(*) filter (where revenue_mismatch)/count(*),2) as mismatch_pct
from clean.fact_sales;""")

,total,mismatches,mismatch_pct
0,60000,59945,99.91


In [ ]:
query("""select count(*) as formula_violations from clean.fact_sales
where revenue_expected <> round(quantity*unit_price*(1-discount_pct),2);""")

,formula_violations
0,0


In [ ]:
query("""select attname, col_description('clean.fact_sales'::regclass, attnum) as comment
from pg_attribute where attrelid = 'clean.fact_sales'::regclass
  and col_description('clean.fact_sales'::regclass, attnum) is not null;""")

,attname,comment
0,revenue,"Source revenue, kept for audit only. Not relia..."
1,units_sold,Quantity in single units: quantity * units_per...
2,revenue_expected,Official net revenue = round(quantity * unit_p...


In [ ]:
query("""select count(*) as no_rule from clean.dim_product where pricing_rule_id is null;""")

,no_rule
0,0


In [ ]:
query("""select count(*) as differs_from_rule
from clean.dim_product p join etl.pricing_rules r on r.rule_id = p.pricing_rule_id
where p.unit_cost <> r.unit_cost or p.list_price <> r.list_price;""")

,differs_from_rule
0,0


In [ ]:
query("""select count(*) as names_with_2_prices from (
  select product_name from clean.dim_product
  group by 1 having count(distinct list_price) > 1) x;""")

,names_with_2_prices
0,0


In [ ]:
query("""select p.product_name, p.unit_cost, p.list_price, r.source
from clean.dim_product p join etl.pricing_rules r on r.rule_id = p.pricing_rule_id
order by p.product_key limit 3;""")

,product_name,unit_cost,list_price,source
0,Smart LED Desk Lamp,82.42,131.87,PLACEHOLDER: median source unit_cost x 1.60
1,Bamboo Cutting Board Set,120.01,192.02,PLACEHOLDER: median source unit_cost x 1.60
2,WiFi 6 Router,58.61,93.78,PLACEHOLDER: median source unit_cost x 1.60
